# HR Candidate Profile Parser

**Goal:** Parse a short resume (CV) snippet into a clean JSON object using LangChain's Output Parser (`StructuredOutputParser`).

**Pipeline:**
1. Take the raw CV text as input.
2. Define the target JSON schema using `ResponseSchema` objects.
3. Build a `StructuredOutputParser` from that schema.
4. Create a prompt that asks the LLM to extract the fields, including the parser's format instructions.
5. Call the LLM and parse its response into a clean Python dict / JSON object.

**Target schema:**
- `full_name` (string)
- `email` (string)
- `education` -> list of {degree, institution, year}
- `skills` -> list of strings
- `experience` -> list of {role, company, years}

## Step 1: Install dependencies

Run this cell once in Colab. It installs LangChain and the Groq integration package. Groq offers a free API key with no credit card required.

**Important:** after this cell finishes, go to the menu **Runtime -> Restart session**, then run all the cells again from the top. This is required so the notebook picks up the newly installed packages.

In [1]:
!pip install -q -U langchain==0.2.16 langchain-groq==0.1.9 langchain-community==0.2.16

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires n

## Step 2: Set your Groq API key

Get a free key at https://console.groq.com/keys (no credit card required). Enter it when prompted. The key is not stored in the notebook file, it is only kept in memory for this session.

In [2]:
import os
import getpass

# If you typed a wrong key before, this clears it so you get prompted again.
if "GROQ_API_KEY" in os.environ:
    del os.environ["GROQ_API_KEY"]

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


## Step 3: The CV text (input)

This is the sample resume snippet from the lab instructions. You can replace this string with any other CV text you want to parse.

In [3]:
cv_text = """
John Smith - john.smith@email.com
Education: B.Sc. Computer Science, MIT, 2020
Skills: Python, Machine Learning, Data Analysis
Experience:
- Software Engineer at Google (2020-2023)
- Data Scientist at OpenAI (2023-Present)
"""

print(cv_text)


John Smith - john.smith@email.com
Education: B.Sc. Computer Science, MIT, 2020
Skills: Python, Machine Learning, Data Analysis
Experience:
- Software Engineer at Google (2020-2023)
- Data Scientist at OpenAI (2023-Present)



## Step 4: Define the output schema

We use LangChain's `ResponseSchema` to describe every field we want back, then combine them into a `StructuredOutputParser`.

In [4]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

response_schemas = [
    ResponseSchema(name="full_name", description="The candidate's full name as a string"),
    ResponseSchema(name="email", description="The candidate's email address as a string"),
    ResponseSchema(
        name="education",
        description=(
            "A list of education entries. Each entry is a JSON object with the keys: "
            "'degree' (string), 'institution' (string), and 'year' (integer)."
        ),
    ),
    ResponseSchema(
        name="skills",
        description="A list of strings, each one a skill mentioned in the CV.",
    ),
    ResponseSchema(
        name="experience",
        description=(
            "A list of work experience entries. Each entry is a JSON object with the keys: "
            "'role' (string), 'company' (string), and 'years' (string, e.g. '2020-2023')."
        ),
    ),
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The candidate's full name as a string
	"email": string  // The candidate's email address as a string
	"education": string  // A list of education entries. Each entry is a JSON object with the keys: 'degree' (string), 'institution' (string), and 'year' (integer).
	"skills": string  // A list of strings, each one a skill mentioned in the CV.
	"experience": string  // A list of work experience entries. Each entry is a JSON object with the keys: 'role' (string), 'company' (string), and 'years' (string, e.g. '2020-2023').
}
```


## Step 5: Build the prompt

The prompt includes the CV text and the format instructions generated by the parser, so the model knows exactly which JSON shape to return.

In [5]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "You are an HR assistant. Extract the candidate information from the CV text below "
    "and return it strictly in the requested format.\n\n"
    "CV text:\n{cv_text}\n\n"
    "{format_instructions}"
)

final_prompt = prompt.format_messages(
    cv_text=cv_text,
    format_instructions=format_instructions,
)

for message in final_prompt:
    print(message.content)

You are an HR assistant. Extract the candidate information from the CV text below and return it strictly in the requested format.

CV text:

John Smith - john.smith@email.com
Education: B.Sc. Computer Science, MIT, 2020
Skills: Python, Machine Learning, Data Analysis
Experience:
- Software Engineer at Google (2020-2023)
- Data Scientist at OpenAI (2023-Present)


The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The candidate's full name as a string
	"email": string  // The candidate's email address as a string
	"education": string  // A list of education entries. Each entry is a JSON object with the keys: 'degree' (string), 'institution' (string), and 'year' (integer).
	"skills": string  // A list of strings, each one a skill mentioned in the CV.
	"experience": string  // A list of work experience entries. Each entry is a JSON object with the keys: 'role' (string),

## Step 6: Call the LLM

In [6]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

response = llm.invoke(final_prompt)
print(response.content)

```json
{
	"full_name": "John Smith",
	"email": "john.smith@email.com",
	"education": [{"degree": "B.Sc. Computer Science", "institution": "MIT", "year": 2020}],
	"skills": ["Python", "Machine Learning", "Data Analysis"],
	"experience": [{"role": "Software Engineer", "company": "Google", "years": "2020-2023"}, {"role": "Data Scientist", "company": "OpenAI", "years": "2023-Present"}]
}
```


## Step 7: Parse the response into clean JSON

In [7]:
import json

parsed_output = output_parser.parse(response.content)

print(json.dumps(parsed_output, indent=2))

{
  "full_name": "John Smith",
  "email": "john.smith@email.com",
  "education": [
    {
      "degree": "B.Sc. Computer Science",
      "institution": "MIT",
      "year": 2020
    }
  ],
  "skills": [
    "Python",
    "Machine Learning",
    "Data Analysis"
  ],
  "experience": [
    {
      "role": "Software Engineer",
      "company": "Google",
      "years": "2020-2023"
    },
    {
      "role": "Data Scientist",
      "company": "OpenAI",
      "years": "2023-Present"
    }
  ]
}


## Step 8: (Optional) Try it with your own CV text

Change the `my_cv_text` variable below to any other resume snippet and run the cell to parse it the same way.

In [8]:
my_cv_text = """
Replace this text with any CV snippet you want to test.
"""

my_prompt = prompt.format_messages(
    cv_text=my_cv_text,
    format_instructions=format_instructions,
)

my_response = llm.invoke(my_prompt)
my_parsed_output = output_parser.parse(my_response.content)

print(json.dumps(my_parsed_output, indent=2))

{
  "full_name": "",
  "email": "",
  "education": "",
  "skills": "",
  "experience": ""
}
